In [19]:
from ultralytics import YOLO
import cv2

In [20]:
model = YOLO("yolov8n.pt")

results = model.predict(
    source="video.mp4",
    save=True,
    conf=0.4,
    device="cuda",
    classes=[0,2,3,5,7]
)

# for i in results[0].boxes:
#     print(i.conf)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/810) c:\Users\ASUS\PycharmProjects\Machine-learning\week3\day2\video.mp4: 384x640 2 cars, 33.3ms
video 1/1 (frame 2/810) c:\Users\ASUS\PycharmProjects\Machine-learning\week3\day2\video.mp4: 384x640 2 cars, 31.3ms
video 1/1 (frame 3/810) c:\Users\ASUS\PycharmProjects\Machine-learning\week3\day2\video.mp4: 384x640 3 cars, 33.5ms
video 1/1 (frame 4/810) c:\Users\ASUS\PycharmProjects\Machine-learning\week3\day2\video.mp4: 384x640 3 cars, 34.1

In [21]:
results[620].show()

In [22]:
r = results[620]

print(r.boxes)
print(r.names)
print(r.orig_shape)
print(r.boxes.conf)

# data -> x1, y1, x2, y2, confidence, classid
# xyxy ->  x1,y1,x2,y2
# xywh -> center and width and height
# xyxyn and xywhn are just normalized versions of the same

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([2., 2., 2., 2., 2., 2.], device='cuda:0')
conf: tensor([0.8221, 0.6641, 0.6384, 0.5438, 0.5239, 0.4543], device='cuda:0')
data: tensor([[6.0877e+02, 1.6747e+02, 7.3295e+02, 2.4752e+02, 8.2209e-01, 2.0000e+00],
        [6.5651e-02, 2.9074e+02, 9.0327e+01, 4.1037e+02, 6.6409e-01, 2.0000e+00],
        [9.0198e+02, 1.1977e+02, 9.4389e+02, 1.4726e+02, 6.3845e-01, 2.0000e+00],
        [9.7160e+02, 1.6995e+02, 1.0678e+03, 3.0562e+02, 5.4383e-01, 2.0000e+00],
        [9.6102e+02, 1.2252e+02, 1.0016e+03, 1.5484e+02, 5.2388e-01, 2.0000e+00],
        [1.0344e+03, 1.2962e+02, 1.0631e+03, 1.5335e+02, 4.5432e-01, 2.0000e+00]], device='cuda:0')
id: None
is_track: False
orig_shape: (720, 1280)
shape: torch.Size([6, 6])
xywh: tensor([[ 670.8594,  207.4984,  124.1843,   80.0497],
        [  45.1961,  350.5569,   90.2609,  119.6307],
        [ 922.9317,  133.5155,   41.9091,   27.4990],
        [1019.6819,  237.7830,   96.1689,  135.6

In [23]:
r = results[620]
x1,y1,x2,y2 = r.boxes.xyxy[5]
print((x2-x1)*(y2-y1), r.boxes.conf)
r.boxes.xyxy

# farther the car leeseer teh area lesser teh confidence

tensor(679.2310, device='cuda:0') tensor([0.8221, 0.6641, 0.6384, 0.5438, 0.5239, 0.4543], device='cuda:0')


tensor([[6.0877e+02, 1.6747e+02, 7.3295e+02, 2.4752e+02],
        [6.5651e-02, 2.9074e+02, 9.0327e+01, 4.1037e+02],
        [9.0198e+02, 1.1977e+02, 9.4389e+02, 1.4726e+02],
        [9.7160e+02, 1.6995e+02, 1.0678e+03, 3.0562e+02],
        [9.6102e+02, 1.2252e+02, 1.0016e+03, 1.5484e+02],
        [1.0344e+03, 1.2962e+02, 1.0631e+03, 1.5335e+02]], device='cuda:0')

In [24]:
for result in results:
    frame = result.plot()  
    cv2.imshow("Frame", frame)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()


<!----------------------------------------Traffic Analysis---------------------------------->

In [25]:
count = {
    
}
for i in results:
    for box in i.boxes:
        cls_name = model.names[int(box.cls)]
        count[cls_name] = count.get(cls_name, 0) + 1
count


{'car': 2680, 'truck': 205, 'person': 21, 'motorcycle': 17}

In [26]:
peak_traffic  = 0
for i in results:
    c = 0
    for box in i.boxes:
        cls_name = model.names[int(box.cls)]
        if cls_name in ['car', 'truck']:
            c += 1
        # count[cls_name] = count.get(cls_name, 0) + 1
    peak_traffic = max(c, peak_traffic)
peak_traffic



10

In [27]:
# custom framing
cap = cv2.VideoCapture("video.mp4")
while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break
    
    results = model(frame)

    result = results[0]
    
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        cls = int(box.cls)
        conf = float(box.conf)
        colors = {
            "person":  (0, 255, 0),
            "car":  (255, 0, 0),
            "truck": (0, 0, 255),
            "motorcycle": (255,0,255)
        }
        label = model.names[cls]
        
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            colors.get(label, (255,255,255)),
            2
        )

        text = f"{label} {conf:.2f}"

        cv2.putText(
            frame,
            text,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )
        
        
        cv2.imshow("Detections", frame)

    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 1 person, 3 cars, 42.4ms
Speed: 5.5ms preprocess, 42.4ms inference, 11.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 39.5ms
Speed: 4.6ms preprocess, 39.5ms inference, 5.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 36.0ms
Speed: 4.7ms preprocess, 36.0ms inference, 5.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 33.3ms
Speed: 3.6ms preprocess, 33.3ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 33.3ms
Speed: 3.5ms preprocess, 33.3ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 cars, 34.2ms
Speed: 3.7ms preprocess, 34.2ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 37.1ms
Speed: 3.0ms preprocess, 37.1ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 33.1ms
Speed: 2.9ms preprocess, 33.1ms inference, 5